In [1]:
from collections import defaultdict
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from groq import Groq  # Make sure groq SDK is installed: pip install groq
from dotenv import load_dotenv
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, ContextTypes, filters
import os

load_dotenv()  # Load variables from .env
api_key = os.getenv("GROQ_API_KEY")
telegram_token = os.getenv("TELEGRAM_TOKEN")

# =========================
# 🔧 Setup: Embeddings & Vector DB
# =========================

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.load_local("panel_db", embeddings=embedding_model, allow_dangerous_deserialization=True)

C:\Users\adarw\AppData\Local\Temp\ipykernel_28512\1003152836.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
c:\Users\adarw\OneDrive\Documents\fypagile\panelAssignment\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# =========================
# 🔧 Setup: Groq API
# =========================

client = Groq(api_key=api_key)  

def query_groq_for_panel_selection(panel_matches: dict, student_title: str, student_area: str) -> str:
    # Format context for LLM
    context = f"Student Project Title: {student_title}\nStudent Project Area: {student_area}\n\n"
    context += "Relevant Panel Data:\n"

    for panel, items in panel_matches.items():
        context += f"\nPanel {panel}:\n"
        for item in items:
            context += f"- {item}\n"

    # Prompt
    prompt = f"""
You are an academic panel recommender system. Based on the student’s title and project area, and the matching history (publication, grants, assignments) of each panel, recommend the top 5 most suitable panels and give a short reason for each.

{context}

Return the result in this format:

1. Panel X - Reason
2. Panel Y - Reason
...
"""

    # Query Groq LLM
    response = client.chat.completions.create(
        model="llama3-70b-8192",
        messages=[
            {"role": "system", "content": "You are an intelligent assistant that recommends academic panels based on expertise."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4
    )

    return response.choices[0].message.content

In [ ]:
# =========================
# 🔁 Main Flow Function
# =========================

def match_student_to_panel(student_title: str, student_area: str) -> str:
    query_text = f"{student_title} - {student_area}"
    query_embedding = embedding_model.embed_query(query_text)

    # Search top 15 most similar docs
    retrieved_docs = db.similarity_search_by_vector(query_embedding, k=15)

    # Group matched documents by panel
    panel_matches = defaultdict(list)
    for doc in retrieved_docs:
        panel = doc.metadata["lecturer_name"]
        source = doc.metadata["source"]
        panel_matches[panel].append(f"[{source.upper()}] {doc.page_content}")

    # Get recommendations from Groq
    result = query_groq_for_panel_selection(panel_matches, student_title, student_area)
    return result

In [4]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    print(f"Start command received from user {update.effective_user.id}")
    await update.message.reply_text("Hi! Send me your project title and area in this format:\n\nTitle - Area")

async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    print(f"Received message: '{update.message.text}' from user {update.effective_user.id}")
    text = update.message.text
    
    if "-" not in text:
        await update.message.reply_text("Please use the format: Title - Area")
        return

    try:
        title, area = [x.strip() for x in text.split("-", 1)]
        print(f"Processing: Title='{title}', Area='{area}'")
        await update.message.reply_text("Finding best panel... 🔍")
        
        result = match_student_to_panel(title, area)
        await update.message.reply_text(result)
        print("Response sent successfully")
        
    except Exception as e:
        print(f"Error processing message: {e}")
        await update.message.reply_text("Sorry, there was an error processing your request. Please try again.")

async def error_handler(update: Update, context: ContextTypes.DEFAULT_TYPE):
    print(f"Update {update} caused error {context.error}")

In [5]:
import asyncio

async def run_bot_jupyter():
    """Alternative method for running in Jupyter notebooks"""
    print("Initializing bot for Jupyter...")
    app = ApplicationBuilder().token(telegram_token).build()

    # Add handlers
    app.add_handler(CommandHandler("start", start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))
    app.add_error_handler(error_handler)

    print("Bot initialized. Starting polling...")
    
    # Initialize the application
    await app.initialize()
    await app.start()
    await app.updater.start_polling()
    
    print("Bot is running! Press Ctrl+C to stop.")
    
    # Keep the bot running
    try:
        while True:
            await asyncio.sleep(1)
    except KeyboardInterrupt:
        print("Stopping bot...")
        await app.updater.stop()
        await app.stop()
        await app.shutdown()


In [6]:
await run_bot_jupyter()

Initializing bot for Jupyter...
Bot initialized. Starting polling...
Bot is running! Press Ctrl+C to stop.
Received message: 'Test' from user 6646295410
Received message: 'Hello' from user 6646295410
Received message: 'Secure File Sharing System Using Advanced Encryption Standard (AES) - Cybersecurity, Data Encryption, Network Security' from user 6646295410
Processing: Title='Secure File Sharing System Using Advanced Encryption Standard (AES)', Area='Cybersecurity, Data Encryption, Network Security'
Response sent successfully


CancelledError: 

In [ ]:
import asyncio

# Debug cell - run this to test your setup
print("=== DEBUGGING INFO ===")
print(f"Telegram token exists: {telegram_token is not None}")
print(f"Telegram token starts with: {telegram_token[:10] if telegram_token else 'None'}...")
print(f"Groq API key exists: {api_key is not None}")
print(f"Database loaded: {db is not None}")

# Test a simple bot first
async def test_simple_bot():
    app = ApplicationBuilder().token(telegram_token).build()
    
    async def simple_start(update: Update, context: ContextTypes.DEFAULT_TYPE):
        print(f"✅ Received /start from {update.effective_user.first_name}")
        await update.message.reply_text("✅ Bot is working!")
    
    async def simple_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
        print(f"✅ Received message: {update.message.text}")
        await update.message.reply_text(f"✅ You said: {update.message.text}")
    
    app.add_handler(CommandHandler("start", simple_start))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, simple_message))
    
    await app.initialize()
    await app.start()
    await app.updater.start_polling()
    
    print("🤖 Simple test bot running...")
    
    try:
        while True:
            await asyncio.sleep(1)
    except KeyboardInterrupt:
        await app.updater.stop()
        await app.stop()
        await app.shutdown()

# Run the simple test
await test_simple_bot()